## 1) Setup Env

In [3]:
from pathlib import Path

ROOT = Path.cwd()
for folder in [
    ROOT / "data" / "raw",
    ROOT / "data" / "processed",
    ROOT / "models",
    ROOT / "src",
    ROOT / "notebooks",
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Root project:", ROOT)
print("Struktur folder siap.")

Root project: d:\TugasUnud\Semester 6\BIG DATA\Analisis Sentimen\notebooks
Struktur folder siap.


In [4]:
import os
import time
from typing import List, Dict, Optional

from dotenv import load_dotenv
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from pymongo import MongoClient

load_dotenv()

YOUTUBE_API_KEY = os.getenv("YOUTUBE_API_KEY", "").strip()
video_ids_raw = os.getenv("YOUTUBE_VIDEO_IDS", "")
YOUTUBE_VIDEO_IDS = [v.strip() for v in video_ids_raw.split(",") if v.strip()]

# Control how many videos to process (default 5). Set NUM_VIDEOS in .env to change.
NUM_VIDEOS = int(os.getenv("NUM_VIDEOS", "5"))
if NUM_VIDEOS < 1:
    raise ValueError("NUM_VIDEOS must be >= 1")
# If more IDs were provided, keep only the first NUM_VIDEOS
if YOUTUBE_VIDEO_IDS:
    YOUTUBE_VIDEO_IDS = YOUTUBE_VIDEO_IDS[:NUM_VIDEOS]

max_comments_raw = os.getenv("MAX_COMMENTS", "").strip()
MAX_COMMENTS = int(max_comments_raw) if max_comments_raw else None

MONGO_URI = os.getenv("MONGO_URI", "").strip()
MONGO_DB = os.getenv("MONGO_DB", "analisis_sentimen").strip()
MONGO_COLLECTION = os.getenv("MONGO_COLLECTION", os.getenv("MONGO_RAW_COMMENTS_COLLECTION", "comments")).strip()

print("API key loaded:", bool(YOUTUBE_API_KEY))
print("Mongo URI loaded:", bool(MONGO_URI))
print("Jumlah video ID (after limit):", len(YOUTUBE_VIDEO_IDS), "(NUM_VIDEOS=" + str(NUM_VIDEOS) + ")")
print("Mode komentar:", "SEMUA TOP-LEVEL, TANPA REPLY")

API key loaded: True
Mongo URI loaded: True
Jumlah video ID (after limit): 5 (NUM_VIDEOS=5)
Mode komentar: SEMUA TOP-LEVEL, TANPA REPLY


In [5]:
import json
import datetime


def append_raw_api_response(
    raw_pages: Optional[List[Dict]],
    endpoint: str,
    request_params: Dict,
    response: Dict,
) -> None:
    if raw_pages is None:
        return

    raw_pages.append(
        {
            "endpoint": endpoint,
            "request_params": request_params,
            "video_id": request_params.get("videoId") or request_params.get("id"),
            "parent_comment_id": request_params.get("parentId"),
            "page_token": request_params.get("pageToken"),
            "fetched_at": datetime.datetime.now(datetime.UTC).isoformat(),
            "response": response,
        }
    )


def fetch_comments(
    api_key: str,
    video_id: str,
    retry: int = 5,
    raw_pages: Optional[List[Dict]] = None,
    raw_comment_threads: Optional[List[Dict]] = None,
) -> List[Dict[str, str]]:
    if not api_key or not video_id:
        raise ValueError("YOUTUBE_API_KEY atau video_id tidak valid")

    youtube = build("youtube", "v3", developerKey=api_key)
    comments: List[Dict[str, str]] = []
    next_page_token = None
    page_count = 0
    seen_comment_ids = set()

    while True:
        res = None
        for attempt in range(retry):
            try:
                request_params = {
                    # include 'replies' so thread-level responses are returned when available
                    "part": "id,snippet,replies",
                    "videoId": video_id,
                    "maxResults": 100,
                    "pageToken": next_page_token,
                    "order": "time",
                    "textFormat": "plainText",
                }
                req = youtube.commentThreads().list(**request_params)
                res = req.execute(num_retries=3)

                append_raw_api_response(raw_pages, "commentThreads.list", request_params, res)
                break
            except (HttpError, TimeoutError, OSError) as err:
                status = getattr(err.resp, "status", None) if isinstance(err, HttpError) else None
                if status == 403:
                    print(f"  ERROR 403 pada video {video_id}. Cek quota/API access.")
                    return comments
                if attempt == retry - 1:
                    print(f"  Gagal request thread page {page_count + 1} pada video {video_id}: {type(err).__name__}: {err}")
                    return comments
                wait_seconds = min(2 ** (attempt + 1), 30)
                print(f"  Retry thread page {page_count + 1} video {video_id} ({attempt + 1}/{retry}) setelah {type(err).__name__}. Tunggu {wait_seconds}s...")
                time.sleep(wait_seconds)

        if res is None:
            break

        items = res.get("items", [])
        if not items:
            break

        page_count += 1

        for item in items:
            if raw_comment_threads is not None:
                raw_thread = dict(item)
                raw_thread["video_id"] = video_id
                raw_comment_threads.append(raw_thread)

            top_obj = item.get("snippet", {}).get("topLevelComment", {})
            top_sn = top_obj.get("snippet", {})
            top_id = top_obj.get("id", "")

            if top_id and top_id not in seen_comment_ids:
                # append top-level comment
                comments.append(
                    {
                        "video_id": video_id,
                        "comment_id": top_id,
                        "parent_comment_id": "",
                        "is_reply": False,
                        "kind": top_obj.get("kind", ""),
                        "etag": top_obj.get("etag", ""),
                        "channel_id": top_sn.get("channelId", ""),
                        "author": top_sn.get("authorDisplayName", ""),
                        "author_channel_id": top_sn.get("authorChannelId", {}).get("value", ""),
                        "author_channel_url": top_sn.get("authorChannelUrl", ""),
                        "author_profile_image_url": top_sn.get("authorProfileImageUrl", ""),
                        "text": top_sn.get("textDisplay", ""),
                        "text_original": top_sn.get("textOriginal", ""),
                        "like_count": top_sn.get("likeCount", 0),
                        "viewer_rating": top_sn.get("viewerRating", ""),
                        "can_rate": top_sn.get("canRate", None),
                        "published_at": top_sn.get("publishedAt", ""),
                        "updated_at": top_sn.get("updatedAt", ""),
                        "total_reply_count": item.get("snippet", {}).get("totalReplyCount", 0),
                        "raw_thread": item,
                        "raw_comment": top_obj,
                        "raw_snippet": top_sn,
                    }
                )
                seen_comment_ids.add(top_id)

                # if replies were returned in the thread payload, flatten and append them
                replies_list = item.get("replies", {}).get("comments", [])
                for reply in replies_list:
                    reply_id = reply.get("id", "")
                    reply_sn = reply.get("snippet", {})
                    if reply_id and reply_id not in seen_comment_ids:
                        comments.append(
                            {
                                "video_id": video_id,
                                "comment_id": reply_id,
                                "parent_comment_id": top_id,
                                "is_reply": True,
                                "kind": reply.get("kind", ""),
                                "etag": reply.get("etag", ""),
                                "channel_id": reply_sn.get("channelId", ""),
                                "author": reply_sn.get("authorDisplayName", ""),
                                "author_channel_id": reply_sn.get("authorChannelId", {}).get("value", ""),
                                "author_channel_url": reply_sn.get("authorChannelUrl", ""),
                                "author_profile_image_url": reply_sn.get("authorProfileImageUrl", ""),
                                "text": reply_sn.get("textDisplay", ""),
                                "text_original": reply_sn.get("textOriginal", ""),
                                "like_count": reply_sn.get("likeCount", 0),
                                "viewer_rating": reply_sn.get("viewerRating", ""),
                                "can_rate": reply_sn.get("canRate", None),
                                "published_at": reply_sn.get("publishedAt", ""),
                                "updated_at": reply_sn.get("updatedAt", ""),
                                "total_reply_count": 0,
                                "raw_thread": item,
                                "raw_comment": reply,
                                "raw_snippet": reply_sn,
                            }
                        )
                        seen_comment_ids.add(reply_id)

        next_page_token = res.get("nextPageToken")
        if not next_page_token:
            break

        time.sleep(0.2)

    print(f"  Video {video_id}: selesai, page thread={page_count}, total top-level comments={len(comments)}")
    return comments


if not YOUTUBE_VIDEO_IDS:
    raise ValueError("Isi YOUTUBE_VIDEO_IDS di .env. Bisa banyak ID dipisahkan koma.")

all_comments = []
all_raw_pages: List[Dict] = []
all_raw_comment_threads: List[Dict] = []
video_metadata = []

print("=" * 60)
print("MULAI MENGAMBIL SEMUA DATA YOUTUBE (TOP-LEVEL COMMENTS ONLY)")
print("=" * 60)

youtube = build("youtube", "v3", developerKey=YOUTUBE_API_KEY)

print("\n[STEP 1] Mengambil metadata video...")
for vid in YOUTUBE_VIDEO_IDS:
    try:
        request_params = {
            "part": "snippet,statistics,contentDetails,status",
            "id": vid,
        }
        req = youtube.videos().list(**request_params)
        res = req.execute(num_retries=3)
        
        append_raw_api_response(all_raw_pages, "videos.list", request_params, res)
        
        if res.get("items"):
            video_info = res["items"][0]
            video_metadata.append({
                "video_id": vid,
                "title": video_info.get("snippet", {}).get("title", ""),
                "description": video_info.get("snippet", {}).get("description", ""),
                "channel_id": video_info.get("snippet", {}).get("channelId", ""),
                "channel_title": video_info.get("snippet", {}).get("channelTitle", ""),
                "published_at": video_info.get("snippet", {}).get("publishedAt", ""),
                "duration": video_info.get("contentDetails", {}).get("duration", ""),
                "view_count": video_info.get("statistics", {}).get("viewCount", 0),
                "like_count": video_info.get("statistics", {}).get("likeCount", 0),
                "comment_count": video_info.get("statistics", {}).get("commentCount", 0),
                "full_response": video_info,
            })
            print(f"  ✓ {vid}: {video_info.get('snippet', {}).get('title', 'Unknown')[:50]}")
        time.sleep(0.3)
    except Exception as e:
        print(f"  ✗ Error fetching video {vid}: {e}")

print("\n[STEP 2] Mengambil semua top-level comments dari semua video...")
for idx, vid in enumerate(YOUTUBE_VIDEO_IDS, 1):
    print(f"\n[{idx}/{len(YOUTUBE_VIDEO_IDS)}] Ambil video: {vid}")
    vid_comments = fetch_comments(
        YOUTUBE_API_KEY,
        vid,
        retry=5,
        raw_pages=all_raw_pages,
        raw_comment_threads=all_raw_comment_threads,
    )
    all_comments.extend(vid_comments)
    print(f"           Terambil: {len(vid_comments)}")

youtube_comment_total_including_replies = sum(int(v.get("comment_count") or 0) for v in video_metadata)
if len(all_comments) != len(all_raw_comment_threads):
    print(f"\nWARNING: top-level comments ({len(all_comments)}) tidak sama dengan raw comment threads ({len(all_raw_comment_threads)}).")
    print("Cek kemungkinan duplicate id atau thread tanpa topLevelComment.")

raw_path_json = ROOT / "data" / "raw" / "comments_raw_all_videos.json"

raw_export = {
    "schema_version": "raw-youtube-api-v1",
    "source": "youtube_data_api_v3",
    "exported_at": datetime.datetime.now(datetime.UTC).isoformat(),
    "request_config": {
        "video_ids": YOUTUBE_VIDEO_IDS,
        "max_comments": None,
        "include_replies": False,
        "comment_order": "time",
        "comment_text_format": "plainText",
    },
    "summary": {
        "total_flattened_comments_seen_for_fetching": len(all_comments),
        "youtube_comment_total_including_replies": youtube_comment_total_including_replies,
        "total_raw_comment_threads": len(all_raw_comment_threads),
        "total_videos": len(video_metadata),
        "total_api_calls_recorded": len(all_raw_pages),
    },
    "raw_comment_threads": all_raw_comment_threads,
    "raw_api_responses": all_raw_pages,
}

with open(raw_path_json, "w", encoding="utf-8") as f:
    json.dump(raw_export, f, ensure_ascii=False, indent=2, default=str)

print(f"\n✓ Raw API JSON lengkap tersimpan: {raw_path_json}")
print(f"  Total API calls: {len(all_raw_pages)}")

if MONGO_URI:
    print(f"\n[MONGODB] Menyimpan semua komentar ke satu collection: {MONGO_DB}.{MONGO_COLLECTION}")
    try:
        client = MongoClient(MONGO_URI)
        db = client[MONGO_DB]
        run_id = raw_export["exported_at"]

        # Do not drop the main `comments` collection to avoid accidental data loss
        old_collections = ["raw_api_responses", "raw_exports", "raw_comment_threads", "videos"]
        for old_collection in old_collections:
            if old_collection != MONGO_COLLECTION and old_collection in db.list_collection_names():
                db.drop_collection(old_collection)
                print(f"  Dropped old collection: {old_collection}")

        raw_comment_records = []
        for index, comment in enumerate(all_comments):
            raw_comment_record = dict(comment)
            raw_comment_record["run_id"] = run_id
            raw_comment_record["schema_version"] = raw_export["schema_version"]
            raw_comment_record["source"] = raw_export["source"]
            raw_comment_record["request_config"] = raw_export["request_config"]
            raw_comment_record["summary"] = raw_export["summary"]
            raw_comment_record["raw_comment_index"] = index
            raw_comment_record["saved_at"] = datetime.datetime.now(datetime.UTC)
            raw_comment_records.append(raw_comment_record)

        db[MONGO_COLLECTION].delete_many({})
        if raw_comment_records:
            db[MONGO_COLLECTION].insert_many(raw_comment_records, ordered=False)

        print(f"  ✓ Inserted {len(raw_comment_records)} raw comments to '{MONGO_COLLECTION}'")
        client.close()
    except Exception as e:
        print(f"  ✗ MongoDB save error: {e}")
else:
    print("\n[MONGODB] MONGO_URI belum diisi; raw hanya tersimpan lokal.")

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"Total comments seen:         {len(all_comments)}")
print(f"Total raw comment threads:   {len(all_raw_comment_threads)}")
print(f"Total videos:                {len(video_metadata)}")
print(f"Total API calls recorded:    {len(all_raw_pages)}")
print(f"\nLocal files saved:")
print(f"  - Raw API JSON only:       {raw_path_json}")
print(f"MongoDB collection:          {MONGO_DB}.{MONGO_COLLECTION if MONGO_URI else '(skipped)'}")
print("=" * 60)

raw_export["raw_api_responses"][0] if raw_export["raw_api_responses"] else raw_export


MULAI MENGAMBIL SEMUA DATA YOUTUBE (TOP-LEVEL COMMENTS ONLY)

[STEP 1] Mengambil metadata video...
  ✓ F6fgLwUeeqI: BATALKAN REVISI UU TNI
  ✓ MxCqHoldj2Y: RUU TNI Resmi Jadi UU! Ada yang Perlu Dikhawatirka
  ✓ sg8Mzx0fZbU: Kenapa UU TNI Banyak Ditolak? Ada Contoh Negara-ne
  ✓ 7CLZkPwhEG4: Revisi UU TNI: Apa dampaknya untuk masyarakat sipi
  ✓ KjXe214MfwQ: RUU TNI itu apa? Mirip Orde Baru? #shorts

[STEP 2] Mengambil semua top-level comments dari semua video...

[1/5] Ambil video: F6fgLwUeeqI
  Video F6fgLwUeeqI: selesai, page thread=35, total top-level comments=3778
           Terambil: 3778

[2/5] Ambil video: MxCqHoldj2Y
  Video MxCqHoldj2Y: selesai, page thread=10, total top-level comments=1126
           Terambil: 1126

[3/5] Ambil video: sg8Mzx0fZbU
  Video sg8Mzx0fZbU: selesai, page thread=23, total top-level comments=2663
           Terambil: 2663

[4/5] Ambil video: 7CLZkPwhEG4
  Video 7CLZkPwhEG4: selesai, page thread=33, total top-level comments=3727
           Terambil: 37

{'endpoint': 'videos.list',
 'request_params': {'part': 'snippet,statistics,contentDetails,status',
  'id': 'F6fgLwUeeqI'},
 'video_id': 'F6fgLwUeeqI',
 'parent_comment_id': None,
 'page_token': None,
 'fetched_at': '2026-05-27T16:20:48.033096+00:00',
 'response': {'kind': 'youtube#videoListResponse',
  'etag': 'ekBnv9bg85A3rZWoDkZk2kuV5NA',
  'items': [{'kind': 'youtube#video',
    'etag': 'L6DrA7PBlGuBzZUzVhV9Mi6L4GQ',
    'id': 'F6fgLwUeeqI',
    'snippet': {'publishedAt': '2025-03-20T01:06:07Z',
     'channelId': 'UCh1SzxJAH9B5nZDYYa42UqQ',
     'title': 'BATALKAN REVISI UU TNI',
     'description': 'Gue membuka youtube membership kepada yang merasa SATU PAHAM, SATU RASA dan SATU JIWA dengan gue:\nhttps://www.youtube.com/channel/UCh1SzxJAH9B5nZDYYa42UqQ/join',
     'thumbnails': {'default': {'url': 'https://i.ytimg.com/vi/F6fgLwUeeqI/default.jpg',
       'width': 120,
       'height': 90},
      'medium': {'url': 'https://i.ytimg.com/vi/F6fgLwUeeqI/mqdefault.jpg',
       'width': 3